# Task 4: Average Salary per Department

## Step 1: Initialize & Load Clean Data

In [1]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

# Load and parse valid records
raw_rdd = sc.textFile("/home/jovyan/data/raw/employees.txt")
header = raw_rdd.first()

def parse_valid(line):
    parts = line.split(",")
    if len(parts) == 9:
        try:
            return [(parts[2], float(parts[4]))]  # (department, salary)
        except ValueError:
            return []
    return []

clean_rdd = raw_rdd.filter(lambda x: x != header) \
                   .filter(lambda x: x.strip() != "") \
                   .flatMap(parse_valid)

print(f"Records loaded: {clean_rdd.count()}")
print("Sample:", clean_rdd.take(3))

Records loaded: 9
Sample: [('Engineering', 125000.0), ('Sales', 85000.0), ('Engineering', 95000.0)]


## Step 2: Calculate Sum & Count per Department

Use `reduceByKey` twice: one for total salary, one for count.

In [2]:
# Sum salaries per department
dept_sum = clean_rdd.reduceByKey(lambda a, b: a + b)

# Count employees per department
dept_count = clean_rdd.mapValues(lambda x: 1) \
                      .reduceByKey(lambda a, b: a + b)

print("Department salary sums:")
for dept, total in dept_sum.collect():
    print(f"  {dept:<12} → ${total:>10,.0f}")

print("\nDepartment employee counts:")
for dept, count in dept_count.collect():
    print(f"  {dept:<12} → {count}")

Department salary sums:
  Engineering  → $   365,000
  Sales        → $   195,000
  Finance      → $   105,000
  IT           → $   115,000
  HR           → $    88,000
  Marketing    → $    92,000

Department employee counts:
  Engineering  → 3
  Sales        → 2
  Finance      → 1
  IT           → 1
  HR           → 1
  Marketing    → 1


## Step 3: Compute Average Salary
Join sum and count, then divide.

In [3]:
# Join sum and count RDDs, then calculate average
dept_avg = dept_sum.join(dept_count).mapValues(lambda x: x[0] / x[1])

print("Average salary per department:\n")
print(f"{'Department':<14} {'Avg Salary':>12} {'Employees':>10}")
print("-" * 38)

# Collect all data and iterate
results = dept_avg.join(dept_count).sortByKey().collect()

for dept, (avg, count) in results:
    print(f"  {dept:<12} ${avg:>10,.0f} {count:>8}")

Average salary per department:

Department       Avg Salary  Employees
--------------------------------------
  Engineering  $   121,667        3
  Finance      $   105,000        1
  HR           $    88,000        1
  IT           $   115,000        1
  Marketing    $    92,000        1
  Sales        $    97,500        2


## Step 4: Sort by Average Salary (Highest First)

In [4]:
# Sort by average salary descending
results = dept_avg.join(dept_count) \
                  .map(lambda x: (x[1][0], (x[0], x[1][1]))) \
                  .sortByKey(ascending=False)

print("Departments ranked by average salary:\n")
rank = 1
for avg, (dept, count) in results.collect():
    bar = "█" * int(avg / 5000)
    print(f"  {rank}. {dept:<12} ${avg:>8,.0f} {bar} ({count} emp)")
    rank += 1

Departments ranked by average salary:

  1. Engineering  $ 121,667 ████████████████████████ (3 emp)
  2. IT           $ 115,000 ███████████████████████ (1 emp)
  3. Finance      $ 105,000 █████████████████████ (1 emp)
  4. Sales        $  97,500 ███████████████████ (2 emp)
  5. Marketing    $  92,000 ██████████████████ (1 emp)
  6. HR           $  88,000 █████████████████ (1 emp)


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35072)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

##  Task 4 Complete

##  Task 4 Complete

**Method:** `reduceByKey` for sum + count → `join` → divide.

**Key finding:** Engineering leads with $121,667 avg salary.